# Credit Card Default Prediction

DTSC 2302, Project 2, Part A

**Team:** Ethan Vanbeurden, Rahul Malla and Eric Testen.

**Rahul Malla's contribution:** Built model pipelines and configured classifier variants, ran stratified five-fold cross-validation and model comparisons, and created evaluation visualizations. Interpretation was completed jointly with teammates.

This notebook presents the initial ten-configuration comparison. See the README for saved results and setup instructions.

AI tools were used for occasional debugging suggestions in the original project.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, average_precision_score, recall_score, classification_report, f1_score, ConfusionMatrixDisplay
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from ydata_profiling import ProfileReport

Inspect the dataset

In [ ]:
from pathlib import Path
DATA_URL = 'https://webpages.charlotte.edu/mschuck1/classes/DTSC2301/Data/credit_card_defaults.csv'
local_data = Path('../data/credit_card_defaults.csv')
ccd = pd.read_csv(local_data if local_data.exists() else DATA_URL)
ccd.dropna(inplace=True)
ccd.columns = ccd.columns.str.strip()
ccd['def_pay_nxt_month'] = ccd.pop('default payment next month')
ccd.info()

### Data dictionary
The dataset describes credit-card customers in Taiwan. It contains 23 original predictors, a customer identifier and the default outcome.

| # | Variable | Type | Description |
|---|---|---|---|
| 1 | `ID` | Categorical | Unique client identifier |
| 2 | `LIMIT_BAL` | Continuous | Amount of given credit in NT dollars (individual + family/supplementary) |
| 3 | `SEX` | Categorical | Gender (1 = male, 2 = female) |
| 4 | `EDUCATION` | Categorical | Education level (1 = graduate school, 2 = university, 3 = high school, 4 = others) |
| 5 | `MARRIAGE` | Categorical | Marital status (1 = married, 2 = single, 3 = others) |
| 6 | `AGE` | Continuous | Age in years |
| 7 | `PAY_0` | Categorical | Repayment status in September 2005 (-1 = on time, 1 = 1 month late, ..., 9 = 9+ months late) |
| 8 | `PAY_2` | Categorical | Repayment status in August 2005 (same scale) |
| 9 | `PAY_3` | Categorical | Repayment status in July 2005 (same scale) |
| 10 | `PAY_4` | Categorical | Repayment status in June 2005 (same scale) |
| 11 | `PAY_5` | Categorical | Repayment status in May 2005 (same scale) |
| 12 | `PAY_6` | Categorical | Repayment status in April 2005 (same scale) |
| 13 | `BILL_AMT1` | Continuous | Bill statement amount in September 2005 (NT dollars) |
| 14 | `BILL_AMT2` | Continuous | Bill statement amount in August 2005 (NT dollars) |
| 15 | `BILL_AMT3` | Continuous | Bill statement amount in July 2005 (NT dollars) |
| 16 | `BILL_AMT4` | Continuous | Bill statement amount in June 2005 (NT dollars) |
| 17 | `BILL_AMT5` | Continuous | Bill statement amount in May 2005 (NT dollars) |
| 18 | `BILL_AMT6` | Continuous | Bill statement amount in April 2005 (NT dollars) |
| 19 | `PAY_AMT1` | Continuous | Amount paid in September 2005 (NT dollars) |
| 20 | `PAY_AMT2` | Continuous | Amount paid in August 2005 (NT dollars) |
| 21 | `PAY_AMT3` | Continuous | Amount paid in July 2005 (NT dollars) |
| 22 | `PAY_AMT4` | Continuous | Amount paid in June 2005 (NT dollars) |
| 23 | `PAY_AMT5` | Continuous | Amount paid in May 2005 (NT dollars) |
| 24 | `PAY_AMT6` | Continuous | Amount paid in April 2005 (NT dollars) |
| 25 | `def_pay_nxt_month` | Categorical | Target variable — defaulted next month (1 = yes, 0 = no) |

In [ ]:
# Optional: enable to generate the large profiling report.
RUN_PROFILE = False
if RUN_PROFILE:
    eda = ProfileReport(ccd, title='Credit Card Defaults', explorative=True)
    eda.to_file('../results/eda_report.html')

**Inspection from EDA Profile Report:**   
- 25 Variables
- 30,000 Records
- 0 Missing cells
- 0 Duplicates

- High correlation between the most recent pay `PAY_0` and  
target variable `def_pay_nxt_month`. A recent pay deliquency is  
pretty telling.
- `BILL_AMT` columns have high correlation with each other which is  
expected.


### Feature preparation
Remove the customer ID, encode demographics and group nonpositive repayment-status codes as zero. Add utilization, total payments and bills, an overall payment-to-bill ratio, and delinquency summaries. Grouping status codes and clipping utilization simplify the data; monthly payment-to-bill ratios are not implemented.

In [ ]:
# Drop ID
ccd.drop('ID', axis=1, inplace=True)

# Encode SEX
ccd['MALE'] = ccd['SEX'].apply(lambda x: 1 if x == 1 else 0)
ccd = ccd.drop('SEX', axis=1)

# Encode EDUCATION
ccd['EDU_GRAD_SCHOOL'] = ccd['EDUCATION'].apply(lambda x: 1 if x == 1 else 0)
ccd['EDU_UNIVERSITY'] = ccd['EDUCATION'].apply(lambda x: 1 if x == 2 else 0)
ccd['EDU_HIGH_SCHOOL'] = ccd['EDUCATION'].apply(lambda x: 1 if x == 3 else 0)
ccd['EDU_OTHER'] = ccd['EDUCATION'].apply(lambda x: 1 if x > 3 or x == 0 else 0)
ccd = ccd.drop('EDUCATION', axis=1)

# Encode MARRIAGE
ccd['MARRIED'] = ccd['MARRIAGE'].apply(lambda x: 1 if x == 1 else 0)
ccd['SINGLE'] = ccd['MARRIAGE'].apply(lambda x: 1 if x == 2 else 0)
ccd['MARRIAGE_OTHER'] = ccd['MARRIAGE'].apply(lambda x: 1 if x > 2 or x == 0 else 0)
ccd = ccd.drop('MARRIAGE', axis=1)

# Fix PAY_0 to PAY_6
for col in ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']:
    ccd[col] = ccd[col].apply(lambda x: 0 if x <= 0 else x)

# Create new features
ccd['CC_UTIL'] = ccd['BILL_AMT1'] / (ccd['LIMIT_BAL'] + 1)
ccd['CC_UTIL'] = ccd['CC_UTIL'].clip(0, 1) 

ccd['TOTAL_PAY'] = ccd['PAY_AMT1'] + ccd['PAY_AMT2'] + ccd['PAY_AMT3'] + ccd['PAY_AMT4'] + ccd['PAY_AMT5'] + ccd['PAY_AMT6']
ccd['TOTAL_BILL'] = ccd['BILL_AMT1'] + ccd['BILL_AMT2'] + ccd['BILL_AMT3'] + ccd['BILL_AMT4'] + ccd['BILL_AMT5'] + ccd['BILL_AMT6']
ccd['OVERALL_PAY_RATIO'] = ccd['TOTAL_PAY'] / (ccd['TOTAL_BILL'] + 1)

ccd['MIN_DELAY'] = ccd[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].min(axis=1)
ccd['MAX_DELAY'] = ccd[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].max(axis=1)
ccd['MED_DELAY'] = ccd[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].median(axis=1)
ccd['NUM_DELAY'] = ccd[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].apply(lambda x: (x > 0).sum(), axis=1)

In [ ]:
ccd.head()

Next, we will train/test split. We'll need to make use of the stratify parameter to handle the response variable imbalance.
The stratify parameter maintains the ratio of our target, since doing it randomly would give different rates.

In [ ]:
#Train-test split
X = ccd.drop('def_pay_nxt_month', axis=1)
y = ccd['def_pay_nxt_month']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=32, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")


Next, we'll build a Pipeline from scikit-learn. This involves a sequence of events where one step's output is the next step's input.
It fits the scaler on `X_train` only (computes mean and standard dev from training data), then Transforms `X_train` using those statistics before it fits the model on the scaled training data.

When you call `pipeline.predict(X_test)`, it:
1. Transforms `X_test` using the training mean and standard dev (not X_test's own statistics)
2. Predicts on the scaled test data

We'll try this on logistic regression first.

In [ ]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        C=1.0,                  # inverse of regularization strength. smaller = more regularization
        class_weight='balanced', # adjusts for the 78/22 target imbalance.
        max_iter=1000,
        random_state=42
    ))
])
lr.fit(X_train, y_train)

Balanced class weights give minority-class observations more weight during training.

Decision Tree (Depth 3): no scaling needed for Decision Tree models since these focus on ordering values.

In [ ]:
dt3 = DecisionTreeClassifier(
    max_depth=3,           # prevents overfitting; unconstrained trees memorize training data
    min_samples_leaf=30,   # minimum samples in a leaf node
    class_weight='balanced',
    random_state=42
)
dt3.fit(X_train, y_train)

Decision Tree (Depth 5)

In [ ]:
dt5 = DecisionTreeClassifier(
    max_depth=5,          
    min_samples_leaf=30,   
    class_weight='balanced',
    random_state=42
)
dt5.fit(X_train, y_train)

Decision Tree (Depth 9)

In [ ]:
dt9 = DecisionTreeClassifier(
    max_depth=9,           
    min_samples_leaf=30,  
    class_weight='balanced',
    random_state=42
)
dt9.fit(X_train, y_train)

Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    class_weight='balanced',
    min_samples_leaf=30,
    random_state=42
)
rf.fit(X_train, y_train)

Linear SVM

In [ ]:
linear_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('lsvm', CalibratedClassifierCV(
        LinearSVC(C=1.0, class_weight='balanced', random_state=42)
    ))
])
linear_svm.fit(X_train, y_train)

RBF Kernel SVM

In [ ]:
rbf_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])
rbf_svm.fit(X_train, y_train)

Linear Discriminant Analysis (LDA)

In [ ]:
lda = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LinearDiscriminantAnalysis(
        solver='svd',    # 'svd' does not compute the full covariance matrix; numerically stable
    ))
])
lda.fit(X_train, y_train)

Quadratic Discriminant Analysis (QDA)

In [ ]:
qda = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', QuadraticDiscriminantAnalysis(
        reg_param=0.1   # regularization for the per-class covariance matrices
    ))
])
qda.fit(X_train, y_train)

XGBoost

In [ ]:
xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    ))
])
xgb.fit(X_train, y_train)

Next, we will evaluate all the models with a focus on Recall.

In [ ]:
def eval(model, X_test, y_test, name, feature_names, top_n=10):
    proba = model.predict_proba(X_test)[:, 1]
    pred  = model.predict(X_test)
    
    print(f"\n{name}")
    print(f"  Avg Precision: {average_precision_score(y_test, proba):.3f}")
    print(f"  Avg Recall: {recall_score(y_test, pred).mean():.3f}")
    print(f"  F1 (default=0.5): {f1_score(y_test, pred):.3f}")
    print(classification_report(y_test, pred, target_names=['No default','Default']))
    confusion_matrix_display = ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=['No default','Default'], cmap='Blues')
    plt.title(f"{name} - Confusion Matrix")
    plt.show()
    
    # Handle both plain models and pipelines
    clf = model[-1] if isinstance(model, Pipeline) else model
    
    if hasattr(clf, 'estimator'):   # CalibratedClassifierCV
        clf = clf.estimator
    
    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
    elif hasattr(clf, 'coef_'):
        importances = abs(clf.coef_[0])
    else:
        print(f"  Top features: not available for this model")
        return
    
    indices = importances.argsort()[::-1][:top_n]
    print(f"\n  Top {top_n} features:")
    for i, idx in enumerate(indices):
        print(f"    {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

### Evaluation objective
Recall is the primary metric in this academic comparison: how many actual defaults are identified? False positives also matter. No monetary cost model was estimated, so the relative cost of errors is an assumption, not a measured business result.

In [ ]:
feature_names = list(X_train.columns)

eval(lr, X_test, y_test, 'Logistic regression', feature_names, top_n=10)
eval(dt3, X_test, y_test, 'Decision tree (max_depth=3)', feature_names, top_n=10)
eval(dt5, X_test, y_test, 'Decision tree (max_depth=5)', feature_names, top_n=10)
eval(dt9, X_test, y_test, 'Decision tree (max_depth=9)', feature_names, top_n=10)
eval(rf, X_test, y_test, 'Random Forest', feature_names, top_n=10)
eval(linear_svm, X_test, y_test, 'Linear SVM', feature_names, top_n=10)
eval(rbf_svm, X_test, y_test, 'RBF kernel SVM', feature_names, top_n=10)
eval(lda, X_test, y_test, 'LDA', feature_names, top_n=10)
eval(qda, X_test, y_test, 'QDA', feature_names, top_n=10)
eval(xgb, X_test, y_test, 'XGBoost', feature_names, top_n=10)

### Initial test results
In the original run, random forest had the highest test recall at 65.7%, with 52.4% F1. The depth-3 tree achieved 64.7% recall and 52.8% F1. The following cross-validation comparison evaluates recall across five training folds.

Cross-Validation of All Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model, name in [(lr, 'LR'), (dt3, 'DT3'), (dt5, 'DT5'), (dt9, 'DT9'), (rf, 'Random Forest'),(linear_svm, 'Linear SVM'), (rbf_svm, 'RBF SVM'),
                    (lda, 'LDA'), (qda, 'QDA'), (xgb, 'XGBoost')]:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='recall', n_jobs=-1)
    print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}")


### Cross-validation results
The depth-3 tree led the original comparison with 68.4% mean recall and a fold standard deviation of 2.4 percentage points.

This was an initial comparison with limited tuning. The tree used balanced class weights; XGBoost's original class_weight argument was ignored and has since been removed. The random forest was restricted to depth 3, and decision thresholds were not tuned. These differences limit conclusions about which model family works best.

Next steps would be to correct XGBoost's weighting and tune settings and thresholds within training-only validation. The original test results had already been inspected, so a stronger final evaluation would use nested cross-validation or a fresh holdout. These extensions have not been run.

### Inspecting the depth-3 tree

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(
    dt3,                    
    feature_names=list(X_train.columns),
    class_names=['No Default', 'Default'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree (max_depth=3)")
plt.tight_layout()
plt.show()

### Interpreting the tree
The first split uses MAX_DELAY, the maximum recorded repayment delay. The tree provides readable rules linking repayment history and financial features to predicted default.

TOTAL_BILL sums monthly statements, which can include carried balances. NUM_DELAY counts observed months with positive delinquency status. The splits show associations; they do not establish customers' income or causes of default. Gini describes weighted class mix within each node.

The team interpreted these findings jointly. Further work would assess false-positive costs, demographic fairness, probability calibration and performance over time.